# Prithvi WxC Downscaling with CORDEX Data: Model Fine-tuning

This notebook demonstrates how to fine-tune the `Prithvi` downscaling model on CORDEX NZ data.

We show how to set up the model, load the pretrained weights, and run finetuning using the CORDEX-derived inputs.

The datasets are from https://zenodo.org/records/17517423

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [ ]:
import os
from pathlib import Path
REPO_ROOT = Path("/mnt/data2/kyo/granite-wxc").resolve()
os.chdir("/mnt/data2/kyo/granite-wxc/examples/CORDEX_ML")
print(f"Working directory set to: {Path.cwd()}")

In [ ]:
# ===================== USER PARAMETERS (EDIT ME) =====================
from nz_params import UserParams, validate_paths


PROJECT_DIR = REPO_ROOT / "examples/CORDEX_ML"
DATASET_ROOT = REPO_ROOT / "granite-geospatial-wxc-downscaling/CORDEX/NZ_domain"

TRAIN_SPLIT = "train/ESD_pseudo_reality"
PREDICTOR_FILE = "ACCESS-CM2_1961-1980_regridded.nc"
TARGET_FILE = "pr_tasmax_ACCESS-CM2_1961-1980.nc"

TRAIN_PREDICTORS = [DATASET_ROOT / TRAIN_SPLIT / "predictors" / PREDICTOR_FILE]
TRAIN_TARGETS = [DATASET_ROOT / TRAIN_SPLIT / "target" / TARGET_FILE]

RUN_NAME_OVERRIDE = 'NZ_T1_ACCESS-CM2_static'  # e.g., 'nz_debug_run'

P = UserParams(
    repo_root=REPO_ROOT,
    project_dir=PROJECT_DIR,
    runs_root=PROJECT_DIR / "runs/NZ_T1_ACCESS-CM2_static_train",
    config_path=PROJECT_DIR / "NZ_T1_ACCESS-CM2_static.yaml",
    run_name=RUN_NAME_OVERRIDE,
    train_predictor_paths=TRAIN_PREDICTORS,
    train_target_paths=TRAIN_TARGETS,
    val_predictor_paths=TRAIN_PREDICTORS,
    val_target_paths=TRAIN_TARGETS,
    test_predictor_paths=TRAIN_PREDICTORS,
    test_target_paths=TRAIN_TARGETS,
    device_target="cuda",
    batch_size=1,
    num_workers=2,
    preferred_checkpoint="best",
)

validate_paths(P)
print(P.summary())


If your current directory is not `granite-wxc/`, change it using the following command:

```bash
%cd <local>/granite-wxc/
```

Replace `<local>` with the appropriate path prefix 

In [ ]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [ ]:
import os
import logging
import warnings

logging.disable(logging.CRITICAL)
warnings.simplefilter(action='ignore', category=FutureWarning)

os.chdir(P.project_dir)
print(f"Project directory set to: {os.getcwd()}")

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler

from cordex_dataset import CordexDownscaleDataset
from cordex_training import (
    get_dataloaders as build_cordex_dataloaders,
    create_finetune_model,
    run_training as launch_cordex_training,
)
from granitewxc.utils.config import get_config
from granitewxc.utils.distributed import init_ddp
from granitewxc.models.model import get_finetune_model_UNET, get_finetune_model
from granitewxc.utils.plot import plot_loss


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from cordex_dataset import CordexDownscaleDataset
from granitewxc.utils.config import get_config
from granitewxc.utils.distributed import init_ddp
from granitewxc.models.model import get_finetune_model_UNET, get_finetune_model
from granitewxc.utils.plot import plot_loss

Run setup utilities from `examples/CORDEX_ML/run_utils.py` are imported below and rely on the repository path specified in the parameter cell.

In [ ]:
torch.jit.enable_onednn_fusion(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
torch.manual_seed(123)

It is possible to use a cpu or gpu/s to generate inferences. Based on avaiablity of a `cuda:gpu`, we set the device that the model uses

In [ ]:
target = (getattr(P, 'device_target', 'auto') or 'auto').lower()
if target == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("device_target is 'cuda' but no CUDA device is available.")
    device = torch.device('cuda')
    use_gpu = True
elif target == 'cpu':
    device = torch.device('cpu')
    use_gpu = False
else:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        use_gpu = True
    else:
        device = torch.device('cpu')
        use_gpu = False


In [ ]:
print(device)

We initialize the Distributed Data Parallel (DDP) environment to leverage multiple GPUs for training

In [ ]:
from pathlib import Path

config_path = P.config_path
config = get_config(str(config_path))
config.device_target = getattr(P, 'device_target', config.device_target)

def _override(attr: str, values):
    if values:
        setattr(config.data, attr, [str(Path(p).resolve()) for p in values])

_override('training_predictor_paths', P.train_predictor_paths)
_override('training_target_paths', P.train_target_paths)
_override('validation_predictor_paths', P.val_predictor_paths)
_override('validation_target_paths', P.val_target_paths)
_override('test_predictor_paths', P.test_predictor_paths)
_override('test_target_paths', P.test_target_paths)

if P.batch_size is not None:
    config.batch_size = int(P.batch_size)
if P.num_workers is not None:
    config.dl_num_workers = int(P.num_workers)

train_dl, val_dl = build_cordex_dataloaders(config, use_gpu)
len(train_dl), len(val_dl)

## Configuration File

The model is configured using `YAML` files.

In these files, you specify:
- Paths to the input data  
- Locations of the pretrained weights  

To ensure compatibility with the provided weights during inference, keep the model configuration consistent with the original definitions 

In [ ]:
import os

os.chdir(P.repo_root)
!pip install -e .
os.chdir(P.project_dir)

In [ ]:
# Configuration already loaded using the USER PARAMETERS block.

In [ ]:
# Working directory managed via the parameter cell.

In [ ]:
from datetime import datetime
from pathlib import Path

from run_utils import (
    copy_scalars_to_run,
    prepare_run_paths,
    snapshot_resolved_config,
    update_manifest,
    write_manifest,
)
from nz_params import resolve_run_dir, export_params

RUN_NAME, resolved_run_dir = resolve_run_dir(P, config)
run_paths = prepare_run_paths(Path(P.runs_root), RUN_NAME)
assert run_paths.run_dir == resolved_run_dir

config.path_experiment = str(run_paths.run_dir)
config.checkpoint_dir = str(run_paths.checkpoints)
config.scalar_dir = str(run_paths.scalars)
config.preproc_dir = str(run_paths.preproc)

scalar_records = copy_scalars_to_run(config, run_paths.scalars)
resolved_config_path = snapshot_resolved_config(
    config, run_paths.config_dir / "resolved.yaml"
)
manifest = write_manifest(
    run_paths,
    run_name=RUN_NAME,
    base_config=str(config_path),
    resolved_config=resolved_config_path,
    scalars=scalar_records,
)

params_json = export_params(P, run_paths.run_dir / "params.json", extra={"run_name": RUN_NAME})
print(f"Run artifacts will be stored in {run_paths.run_dir}")
print(f"Config snapshot: {resolved_config_path}")
print(f"Saved parameter snapshot to {params_json}")

## Dataloader 

In this CORDEX example we will use a single regridded data pair for both training and validation.

In [ ]:
# This cell is deprecated; please use the helper functions defined earlier for dataloaders.


### Sample shapes

We check the batched tensor shapes from the CORDEX dataloader.

In [ ]:
example_batch = next(iter(train_dl))
example_batch["x"].shape, example_batch["y"].shape, example_batch["static_x"].shape

In [ ]:
target_shape = tuple(example_batch["y"].shape)
var_names = list(config.data.output_vars)
print(f"Configured output variables: {var_names}")
print(f"Sample target tensor shape: {target_shape}")
if target_shape[1] != len(var_names):
    raise ValueError(
        f"Dataloader returned {target_shape[1]} channels but config specifies {len(var_names)}"
    )
missing_expected = {"pr", "tasmax"} - set(var_names)
if missing_expected:
    raise ValueError(f"Missing expected NZ targets: {sorted(missing_expected)}")
print("Fine-tuning includes both 'pr' and 'tasmax'.")


## Model Initialization

We provide **2** different model architectures `UNET-like` and `CONV` 

Both architectures include:  
1. **Patch Embedding**: Extracts shallow features from the input data  
2. **Feature Extraction**: Utilizes the Prithvi backbone to extract deeper features  

The key difference is that the UNET-like version incorporates **static high-resolution data** into the model

In this notebook, we use the **UNET-like** version

To switch to the **CONV** model, update the configuration file accordingly and use `get_finetune_model(config)` 

In [ ]:
model = create_finetune_model(config).to(device)


We can now load the pretrained weights

In [ ]:
# Pretrained weights are loaded via create_finetune_model().


## Finetuning

The model is now ready for training

In [ ]:
from torch.optim import AdamW
from torch.cuda.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from granitewxc.utils.trainer import train_model
from granitewxc.models.loss import rmse_loss

## 
Defining the optimizer, scaler, and scheduler for the training process.

In [ ]:
# Optimizer configuration handled inside cordex_training.run_training.


We'll train the model using just 1 data pair and run it for only 5 epochs

Let's train it for a couple of epochs based on the configuration file

In [ ]:
try:
    train_losses, val_losses = launch_cordex_training(config, num_gpus=torch.cuda.device_count(), save_every=5)
except Exception as exc:
    print(f"Training failed: {exc}")
    raise


In [ ]:
from pathlib import Path

checkpoint_info = {
    "best": str(Path(config.checkpoint_dir) / "best.ckpt"),
    "last": str(Path(config.checkpoint_dir) / "last.ckpt"),
}
manifest = update_manifest(run_paths.run_dir, checkpoints=checkpoint_info)
print("Updated run manifest with checkpoint locations:")
for label, ckpt_path in checkpoint_info.items():
    print(f"  {label}: {ckpt_path}")


In [ ]:
plot_loss(train_losses, val_losses)